In [ ]:
import requests
import pandas as pd
import numpy as np
import datetime
import os
from google.colab import drive

# Options d'affichage Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


In [ ]:
# URL du dump statique fourni par IBM
static_json_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

response = requests.get(static_json_url)
if response.status_code == 200:
    data = response.json()
    df_raw = pd.json_normalize(data)

    # Restreindre aux lancements Falcon 9 (optionnel selon ton avancement dans le lab)
    # df_raw = df_raw[df_raw['cores'].map(len) == 1] # Exemple de filtre du lab
    print(f"✅ Données brutes chargées. Forme du dataset : {df_raw.shape}")
else:
    print("❌ Échec du téléchargement des données statiques.")

✅ Données brutes chargées. Forme du dataset : (107, 42)


In [ ]:
# --- CELLULE : FONCTIONS DE PARSING OFFICIELLES ---

def getBoosterVersion(data):
    # Dictionnaire de secours au cas où l'API v4 est en berne
    fallback = {
        '5e9d0d95eda69955f709d1eb': 'Falcon 1',
        '5e9d0d95eda69973a809d1ec': 'Falcon 9',
        '5e9d0d95eda69974db09d1ed': 'Falcon Heavy'
    }
    for x in data['rocket']:
        if x:
            try:
                # Logique officielle IBM v4
                response = requests.get(f"https://api.spacexdata.com/v4/rockets/{x}").json()
                BoosterVersion.append(response['name'])
            except:
                # Sécurité "Cloud-Native" : évite le crash global
                BoosterVersion.append(fallback.get(x, "Unknown Rocket"))

def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            try:
                response = requests.get(f"https://api.spacexdata.com/v4/launchpads/{x}").json()
                Longitude.append(response['longitude'])
                Latitude.append(response['latitude'])
                LaunchSite.append(response['name'])
            except:
                # Secours si l'API Launchpads ne répond pas
                Longitude.append(None)
                Latitude.append(None)
                LaunchSite.append("Unknown Pad")

def getPayloadData(data):
    # Version simplifiée et sécurisée de l'extraction des payloads (image_d755e3.png)
    for load in data['payloads']:
        if load:
            try:
                # Dans le JSON statique, 'payloads' est une liste d'IDs (on prend le premier ID)
                payload_id = load[0] if isinstance(load, list) else load
                response = requests.get(f"https://api.spacexdata.com/v4/payloads/{payload_id}").json()
                PayloadMass.append(response['mass_kg'])
                Orbit.append(response['orbit'])
            except:
                PayloadMass.append(None)
                Orbit.append(None)

def getCoreData(data):
    # Extraction des données du premier étage (image_d75626.png)
    for core in data['cores']:
        if core and len(core) > 0:
            c = core[0] # Premier core
            try:
                response = requests.get(f"https://api.spacexdata.com/v4/cores/{c['core']}").json()
                Block.append(response['block'])
                ReusedCount.append(response['reuse_count'])
                Serial.append(response['serial'])
            except:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)

            # Données provenant directement de la table 'launches'
            Outcome.append(str(c['landing_success']) + ' ' + str(c['landing_type']))
            Flights.append(c['flight'])
            GridFins.append(c['gridfins'])
            Reused.append(c['reused'])
            Legs.append(c['legs'])
            LandingPad.append(c['landpad'])
        else:
            # Remplissage par défaut si pas de core disponible
            for lst in [Block, ReusedCount, Serial, Outcome, Flights, GridFins, Reused, Legs, LandingPad]:
                lst.append(None)

In [ ]:
# --- CELLULE : INITIALISATION ET SCRIPT D'EXÉCUTION ---

# 1. Initialisation de toutes les listes globales exigées par le Lab
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

# 2. Filtrage des données brutes (Étape cruciale du TP d'IBM)
# Conversion de la date brute en objet datetime.date pour appliquer le filtre
df_raw['date'] = pd.to_datetime(df_raw['date_utc']).dt.date

# On filtre pour exclure les données post-13 Novembre 2020
df_filtered = df_raw[df_raw['date'] <= datetime.date(2020, 11, 13)].copy()

# On filtre pour n'avoir que les Falcon 9 en utilisant l'ID de la fusée
df_filtered = df_filtered[df_filtered['rocket'] == '5e9d0d95eda69973a809d1ec'].copy()

# On filtre pour n'avoir que les tirs à 1 seul core
df_filtered = df_filtered[df_filtered['cores'].map(len) == 1]

# Nettoyage des lignes sans fusées valides
df_filtered = df_filtered.dropna(subset=['rocket']).reset_index(drop=True)

print(f"Structure après filtrage des dates et de la fusée : {df_filtered.shape} lignes.")

# 3. Exécution séquentielle des fonctions de parsing (image_d759e5.png)
print("Parsing en cours (cette étape fait des appels HTTP et peut prendre 1 à 2 minutes)...")
getBoosterVersion(df_filtered)
getLaunchSite(df_filtered)
getPayloadData(df_filtered)
getCoreData(df_filtered)
print("🎉 Extraction terminée avec succès !")

Structure après filtrage des dates et de la fusée : (98, 43) lignes.
Parsing en cours (cette étape fait des appels HTTP et peut prendre 1 à 2 minutes)...
🎉 Extraction terminée avec succès !


In [ ]:
# --- CELLULE : ASSEMBLAGE ET EXPORT ---

launch_dict = {
    'FlightNumber': list(range(1, len(df_filtered) + 1)),
    'Date': df_filtered['date'].tolist(),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}

# Création du DataFrame final d'analyse
df_final = pd.DataFrame(launch_dict)

# Sauvegarde propre dans ta dssandbox locale/cloud
df_final.to_csv('dataset_part_1.csv', index=False)
print("💾 Fichier final sauvegardé dans dssandbox !")
df_final.head()

💾 Fichier final sauvegardé dans dssandbox !


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,None,None,Unknown Pad,None None,1,False,False,False,None,None,None,None,None,None
1,2,2010-12-08,Falcon 9,None,None,Unknown Pad,None None,1,False,False,False,None,None,None,None,None,None
2,3,2012-05-22,Falcon 9,None,None,Unknown Pad,None None,1,False,False,False,None,None,None,None,None,None
3,4,2012-10-08,Falcon 9,None,None,Unknown Pad,None None,1,False,False,False,None,None,None,None,None,None
4,5,2013-03-01,Falcon 9,None,None,Unknown Pad,None None,1,False,False,False,None,None,None,None,None,None


In [ ]:
df_final.shape

(98, 17)